# LeetCode 1186 — Maximum Subarray Sum with One Deletion bằng Python

Link đề: https://leetcode.com/problems/maximum-subarray-sum-with-one-deletion/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming

Skeleton LeetCode:

```python
class Solution:
    def maximumSum(self, arr: list[int]) -> int:
        ...
```

Ghi chú: đây là bản nâng cấp của bài kinh điển **Maximum Subarray (LeetCode 53, thuật toán Kadane)**. Điểm mới: được **xóa tối đa một phần tử** trong đoạn con, nhưng đoạn còn lại **không được rỗng**. Lời giải đẹp nhất là **DP hai trạng thái** (chưa xóa / đã xóa một lần) chạy `O(n)` thời gian, `O(1)` bộ nhớ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chọn một **subarray** (đoạn con liên tiếp) không rỗng, được bỏ đi **không quá một** phần tử, sao cho tổng phần còn lại (vẫn không rỗng) lớn nhất.
2. Ôn lại **Kadane**: `best_end_here = max(x, best_end_here + x)`.
3. Học cách **thêm một chiều trạng thái** vào DP: `keep` (chưa xóa) và `drop` (đã xóa đúng một phần tử).
4. Biết lời giải thay thế bằng **Kadane xuôi + Kadane ngược** (prefix/suffix) và cách ghép tại phần tử bị xóa.
5. Tránh các bẫy: mảng toàn số âm, khởi tạo `drop` sai cho phần tử đầu tiên, trả về `0` khi không được phép rỗng.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `arr`. Trả về **tổng lớn nhất** của một đoạn con liên tiếp **không rỗng**, trong đó bạn được (tùy chọn) xóa **tối đa một** phần tử. Sau khi xóa, đoạn phải còn **ít nhất một** phần tử.

**Ví dụ 1:** `arr = [1,-2,0,3]` → `4` (chọn cả mảng, xóa `-2`, còn `[1,0,3]` tổng `4`).

**Ví dụ 2:** `arr = [1,-2,-2,3]` → `3` (chỉ chọn `[3]`).

**Ví dụ 3:** `arr = [-1,-1,-1,-1]` → `-1` (không được chọn `[-1]` rồi xóa nó để có tổng `0`, vì đoạn rỗng).

**Ràng buộc:** `1 <= arr.length <= 10^5`, `-10^4 <= arr[i] <= 10^4`.

In [1]:
import random
import time
from itertools import accumulate
from typing import List

examples = [
    ([1, -2, 0, 3], 4),
    ([1, -2, -2, 3], 3),
    ([-1, -1, -1, -1], -1),
]
for arr, want in examples:
    print(arr, "->", want)

[1, -2, 0, 3] -> 4
[1, -2, -2, 3] -> 3
[-1, -1, -1, -1] -> -1


## 3. Kiến thức nền

### 3.1 Subarray (đoạn con liên tiếp) và cắt list

Subarray là các phần tử **liền nhau** `arr[l..r]`. Trong Python lấy bằng `arr[l:r+1]` (đầu mút phải không tính). Một mảng dài `n` có `n(n+1)/2` subarray.

In [2]:
arr = [1, -2, 0, 3]
subs = [arr[l:r + 1] for l in range(len(arr)) for r in range(l, len(arr))]
print(len(subs), subs[:5])

10 [[1], [1, -2], [1, -2, 0], [1, -2, 0, 3], [-2]]


### 3.2 Thuật toán Kadane (bài 53)

Gọi `cur` = tổng lớn nhất của subarray **kết thúc tại** vị trí hiện tại. Khi gặp `x`: hoặc nối `x` vào đoạn cũ (`cur + x`), hoặc bắt đầu lại từ `x`. Vậy `cur = max(x, cur + x)`. Đáp án là `max` của mọi `cur`.

In [3]:
def kadane(a):
    cur = best = a[0]
    for x in a[1:]:
        cur = max(x, cur + x)
        best = max(best, cur)
    return best


print(kadane([1, -2, 0, 3]), kadane([-3, -1, -2]), kadane([2, -1, 2]))
assert kadane([-3, -1, -2]) == -1

3 -1 3


### 3.3 `float('-inf')` làm giá trị "không hợp lệ"

Một trạng thái **chưa thể tồn tại** (ví dụ "đã xóa một phần tử" khi mới đứng ở phần tử đầu tiên — xóa nó thì đoạn rỗng) nên gán `-inf`. Phép `max` sẽ không bao giờ chọn nó, và `-inf + x` vẫn là `-inf`.

In [4]:
NEG = float('-inf')
print(max(NEG, 5), NEG + 10**4, max(NEG + 3, -7))

5 -inf -7


### 3.4 Gán đồng thời `a, b = ..., ...`

Khi trạng thái mới phụ thuộc vào **giá trị cũ** của nhiều biến, dùng gán đồng thời để vế phải được tính hết bằng giá trị cũ trước khi ghi.

In [5]:
keep, drop = 5, 2
keep, drop = keep + 1, keep   # drop nhận keep CŨ = 5
print(keep, drop)
assert (keep, drop) == (6, 5)

6 5


### 3.5 `itertools.accumulate` — tổng tiền tố (prefix sum)

`list(accumulate(a, initial=0))` cho `P` với `P[i] = a[0] + ... + a[i-1]`; tổng đoạn `a[l..r] = P[r+1] - P[l]`. Hữu ích cho brute force `O(n^2)`.

In [6]:
a = [1, -2, 0, 3]
P = list(accumulate(a, initial=0))
print(P, "tổng a[1..3] =", P[4] - P[1])

[0, 1, -1, -1, 2] tổng a[1..3] = 1


## 4. Ý tưởng nền tảng / trực giác

Với `arr = [1, -2, 0, 3]`:

- Không xóa: Kadane cho `3` (đoạn `[0, 3]` hoặc `[3]`).
- Xóa `-2` trong đoạn `[1, -2, 0, 3]`: `1 + 0 + 3 = 4`.

Nhìn theo cách khác: nếu xóa phần tử ở vị trí `i`, đoạn còn lại = **(đoạn kết thúc tại `i-1`)** + **(đoạn bắt đầu tại `i+1`)**. Muốn tổng lớn nhất thì mỗi nửa cũng phải lớn nhất:

| i (bị xóa) | tốt nhất kết thúc tại i-1 | tốt nhất bắt đầu tại i+1 | tổng |
|---|---|---|---|
| 1 (`-2`) | 1 | 3 (`[0,3]`) | **4** |
| 2 (`0`) | -1 (`[1,-2]`) hoặc -2 → -1 | 3 | 2 |

→ Đáp án `max(3, 4) = 4`.

In [7]:
arr = [1, -2, 0, 3]
for i in range(1, len(arr) - 1):
    left_best = max(sum(arr[l:i]) for l in range(i))
    right_best = max(sum(arr[i + 1:r + 1]) for r in range(i + 1, len(arr)))
    print(f"xóa i={i} ({arr[i]}): trái={left_best}, phải={right_best}, tổng={left_best + right_best}")

xóa i=1 (-2): trái=1, phải=3, tổng=4
xóa i=2 (0): trái=-1, phải=3, tổng=2


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** thử mọi đoạn `[l, r]`. Với đoạn đó, hai lựa chọn:
- Không xóa: tổng `S`.
- Xóa một phần tử (chỉ khi độ dài `≥ 2`): xóa phần tử **nhỏ nhất** là tốt nhất → `S - min(đoạn)`.

Duy trì `S` và `min` khi mở rộng `r` để được `O(n^2)` thay vì `O(n^3)`.

In [8]:
def brute(arr: List[int]) -> int:
    n = len(arr)
    best = float('-inf')
    for l in range(n):
        s, mn = 0, float('inf')
        for r in range(l, n):
            s += arr[r]
            mn = min(mn, arr[r])
            best = max(best, s)
            if r > l:
                best = max(best, s - mn)
    return best


for arr, want in examples:
    print(brute(arr), want)
    assert brute(arr) == want

4 4
3 3
-1 -1


Một brute force "thô" hơn nữa (`O(n^3)`) — liệt kê đoạn, liệt kê phần tử bị xóa — dùng làm oracle tuyệt đối đáng tin cho mảng rất nhỏ:

In [9]:
def brute_raw(arr):
    n = len(arr)
    best = float('-inf')
    for l in range(n):
        for r in range(l, n):
            seg = arr[l:r + 1]
            best = max(best, sum(seg))
            if len(seg) >= 2:
                for k in range(len(seg)):
                    best = max(best, sum(seg) - seg[k])
    return best


for arr, want in examples:
    assert brute_raw(arr) == want
print("brute_raw OK")

brute_raw OK


**Độ phức tạp:** `O(n^2)` thời gian, `O(1)` bộ nhớ. Với `n = 10^5` là `5·10^9` bước → **TLE**.

**Nút thắt:** ta tính lại tổng của các đoạn chồng lên nhau rất nhiều lần. Kadane đã dạy: chỉ cần nhớ "tốt nhất kết thúc tại đây" là đủ. Câu hỏi: có thể nhớ thêm thông tin "đã dùng quyền xóa hay chưa" không?

In [10]:
random.seed(0)
for n in [500, 1000, 2000]:
    a = [random.randint(-10**4, 10**4) for _ in range(n)]
    t = time.perf_counter(); brute(a); print(n, f"{time.perf_counter() - t:.3f}s")

500 0.019s
1000 0.079s


2000 0.278s


## 6. Bước đột phá

Định nghĩa hai trạng thái tại vị trí `i` (đoạn **phải kết thúc tại `i`**):

- `keep[i]` = tổng lớn nhất của đoạn kết thúc tại `i`, **chưa xóa** gì. (Chính là Kadane.)
- `drop[i]` = tổng lớn nhất của đoạn kết thúc tại `i`, **đã xóa đúng một** phần tử (và còn ít nhất một phần tử).

**Chuyển trạng thái:**
- `keep[i] = max(arr[i], keep[i-1] + arr[i])`.
- `drop[i] = max(drop[i-1] + arr[i], keep[i-1])`:
  - **Nếu** quyền xóa đã dùng trước đó → nối `arr[i]` vào: `drop[i-1] + arr[i]`.
  - **Nếu** xóa chính `arr[i]` → phần còn lại là đoạn kết thúc tại `i-1` chưa xóa: `keep[i-1]` (không rỗng vì `keep` luôn là đoạn không rỗng).
- Khởi tạo: `keep[0] = arr[0]`, `drop[0] = -inf` (xóa `arr[0]` sẽ thành rỗng).

**Đáp án** `= max over i của max(keep[i], drop[i])`.

Chú ý: `drop` **không** có lựa chọn "bắt đầu lại từ `arr[i]`" vì đoạn bắt đầu tại `i` chưa thể có phần tử bị xóa nằm trước nó (xóa phần tử đầu đoạn thì tương đương đoạn ngắn hơn chưa xóa, đã được `keep` bao).

## 7. Các cách tiếp theo

### Cách 2 — Thử xóa từng vị trí, chạy Kadane mỗi lần

Ý tưởng cải tiến nhẹ: đáp án = `max(kadane(arr), max_i kadane(arr bỏ i))`. Nhưng chú ý: "bỏ `i`" phải **ghép** hai nửa lại — tương đương Kadane trên mảng đã xóa phần tử `i`. Vẫn `O(n^2)` nhưng code rất đơn giản và trực quan.

In [11]:
def solve_delete_each(arr: List[int]) -> int:
    best = kadane(arr)
    if len(arr) >= 2:
        for i in range(len(arr)):
            best = max(best, kadane(arr[:i] + arr[i + 1:]))
    return best


for arr, want in examples:
    assert solve_delete_each(arr) == want
print("Cách 2 OK")

Cách 2 OK


**Vì sao đúng?** Mọi đoạn con của mảng đã xóa `i` mà **chứa vị trí ghép** chính là một đoạn gốc có xóa `arr[i]`; đoạn không chứa vị trí ghép là đoạn gốc không xóa. Ngược lại mọi phương án hợp lệ đều xuất hiện. **Độ phức tạp:** `O(n^2)`.

### Cách 3 — Kadane xuôi + Kadane ngược (prefix/suffix DP)

- `f[i]` = tổng lớn nhất của đoạn **kết thúc** tại `i` (Kadane xuôi).
- `g[i]` = tổng lớn nhất của đoạn **bắt đầu** tại `i` (Kadane ngược).
- Không xóa: `max(f)`. Xóa phần tử `i` (với `0 < i < n-1`): `f[i-1] + g[i+1]`.

Xóa phần tử ở **mép** đoạn không bao giờ có lợi hơn việc chọn đoạn ngắn hơn, nên chỉ cần `i` ở giữa có hai nửa không rỗng.

In [12]:
def solve_prefix_suffix(arr: List[int]) -> int:
    n = len(arr)
    f = arr[:]
    for i in range(1, n):
        f[i] = max(arr[i], f[i - 1] + arr[i])
    g = arr[:]
    for i in range(n - 2, -1, -1):
        g[i] = max(arr[i], g[i + 1] + arr[i])
    best = max(f)
    for i in range(1, n - 1):
        best = max(best, f[i - 1] + g[i + 1])
    return best


for arr, want in examples:
    assert solve_prefix_suffix(arr) == want
print(solve_prefix_suffix([1, -2, 0, 3]))

4


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ. Ưu điểm: dễ giải thích, dễ mở rộng sang "xóa đúng một phần tử tại vị trí tùy ý" hay "xóa một đoạn con".

### Cách 4 — DP hai trạng thái với mảng `keep[]`, `drop[]`

Cài đặt trực tiếp công thức ở mục 6 bằng mảng để dễ quan sát.

In [13]:
def solve_dp_arrays(arr: List[int]) -> int:
    n = len(arr)
    NEG = float('-inf')
    keep = [0] * n
    drop = [NEG] * n
    keep[0] = arr[0]
    for i in range(1, n):
        keep[i] = max(arr[i], keep[i - 1] + arr[i])
        drop[i] = max(drop[i - 1] + arr[i], keep[i - 1])
    return max(max(keep), max(drop))


for arr, want in examples:
    assert solve_dp_arrays(arr) == want
print(solve_dp_arrays([1, -2, 0, 3]))

4


### Cách 5 — DP hai trạng thái, bộ nhớ `O(1)` (lời giải chuẩn)

Vì `keep[i]`, `drop[i]` chỉ phụ thuộc `keep[i-1]`, `drop[i-1]` nên chỉ cần hai biến. **Thứ tự cập nhật:** `drop` dùng `keep` **cũ** → cập nhật `drop` trước hoặc dùng gán đồng thời.

In [14]:
def solve_dp(arr: List[int]) -> int:
    keep, drop = arr[0], float('-inf')
    best = arr[0]
    for x in arr[1:]:
        keep, drop = max(x, keep + x), max(drop + x, keep)
        best = max(best, keep, drop)
    return best


for arr, want in examples:
    assert solve_dp(arr) == want
print([solve_dp(a) for a, _ in examples])

[4, 3, -1]


**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ.

## 8. TRACE từng bước

Trace Cách 5 trên `arr = [1, -2, 0, 3]`:

| i | x | keep (mới) | drop (mới) = max(drop+x, keep cũ) | best |
|---|---|---|---|---|
| 0 | 1 | 1 | -inf | 1 |
| 1 | -2 | max(-2, -1) = -1 | max(-inf, 1) = 1 | 1 |
| 2 | 0 | max(0, -1) = 0 | max(1+0, -1) = 1 | 1 |
| 3 | 3 | max(3, 3) = 3 | max(1+3, 0) = **4** | **4** |

`drop = 4` ở bước cuối tương ứng đoạn `[1, -2, 0, 3]` đã xóa `-2`.

In [15]:
def trace_dp(arr):
    keep, drop = arr[0], float('-inf')
    best = arr[0]
    print(f"i=0 x={arr[0]:>3} keep={keep:>5} drop={drop!s:>5} best={best}")
    for i in range(1, len(arr)):
        x = arr[i]
        new_keep = max(x, keep + x)
        new_drop = max(drop + x, keep)
        why = "xóa x" if new_drop == keep else "nối x"
        keep, drop = new_keep, new_drop
        best = max(best, keep, drop)
        print(f"i={i} x={x:>3} keep={keep:>5} drop={drop!s:>5} ({why}) best={best}")
    return best


trace_dp([1, -2, 0, 3])
print()
trace_dp([-1, -1, -1, -1])

i=0 x=  1 keep=    1 drop= -inf best=1
i=1 x= -2 keep=   -1 drop=    1 (xóa x) best=1
i=2 x=  0 keep=    0 drop=    1 (nối x) best=1
i=3 x=  3 keep=    3 drop=    4 (nối x) best=4

i=0 x= -1 keep=   -1 drop= -inf best=-1
i=1 x= -1 keep=   -1 drop=   -1 (xóa x) best=-1
i=2 x= -1 keep=   -1 drop=   -1 (xóa x) best=-1
i=3 x= -1 keep=   -1 drop=   -1 (xóa x) best=-1


-1

In [16]:
def trace_prefix_suffix(arr):
    n = len(arr)
    f = arr[:]
    for i in range(1, n):
        f[i] = max(arr[i], f[i - 1] + arr[i])
    g = arr[:]
    for i in range(n - 2, -1, -1):
        g[i] = max(arr[i], g[i + 1] + arr[i])
    print("arr =", arr)
    print("f   =", f, "(tốt nhất kết thúc tại i)")
    print("g   =", g, "(tốt nhất bắt đầu tại i)")
    for i in range(1, n - 1):
        print(f"  xóa i={i}: f[{i-1}] + g[{i+1}] = {f[i-1]} + {g[i+1]} = {f[i-1] + g[i+1]}")


trace_prefix_suffix([1, -2, 0, 3])

arr = [1, -2, 0, 3]
f   = [1, -1, 0, 3] (tốt nhất kết thúc tại i)
g   = [2, 1, 3, 3] (tốt nhất bắt đầu tại i)
  xóa i=1: f[0] + g[2] = 1 + 3 = 4
  xóa i=2: f[1] + g[3] = -1 + 3 = 2


## 9. Code nộp LeetCode

In [17]:
class Solution:
    def maximumSum(self, arr: List[int]) -> int:
        keep = best = arr[0]          # keep: đoạn kết thúc tại i, chưa xóa
        drop = float('-inf')          # drop: đoạn kết thúc tại i, đã xóa 1 phần tử
        for x in arr[1:]:
            keep, drop = max(x, keep + x), max(drop + x, keep)
            best = max(best, keep, drop)
        return best


sol = Solution()
for arr, want in examples:
    print(sol.maximumSum(arr), want)
    assert sol.maximumSum(arr) == want

4 4
3 3
-1 -1


## 10. Vì sao đúng

**Mệnh đề (quy nạp theo `i`):** sau khi xử lý `arr[i]`, `keep` là tổng lớn nhất của đoạn không rỗng kết thúc tại `i` không xóa, `drop` là tổng lớn nhất của đoạn kết thúc tại `i` đã xóa đúng một phần tử và còn ≥ 1 phần tử (hoặc `-inf` nếu không tồn tại).

- **Cơ sở `i = 0`:** đoạn duy nhất là `[arr[0]]` → `keep = arr[0]`; xóa nó thì rỗng → `drop = -inf`. ✓
- **Bước `i`:** xét đoạn tối ưu kết thúc tại `i` đã xóa một phần tử. Có hai khả năng loại trừ nhau:
  1. Phần tử bị xóa là `arr[i]`: phần còn lại là đoạn **không rỗng** kết thúc tại `i-1`, không xóa → tốt nhất `keep[i-1]`.
  2. Phần tử bị xóa nằm trước `i`: bỏ `arr[i]` đi thì còn đoạn kết thúc tại `i-1` đã xóa một phần tử (không rỗng, vì nếu rỗng thì đoạn chỉ gồm phần tử bị xóa và `arr[i]`, tức là trường hợp xóa phần tử đầu — tương đương đoạn `[arr[i]]`, đã được `keep` xét). → tốt nhất `drop[i-1] + arr[i]`.

  Lấy `max` hai khả năng ra đúng `drop[i]`. `keep[i]` là Kadane. ✓

Mọi đoạn hợp lệ đều kết thúc tại một `i` nào đó, nên `max` của tất cả `keep`, `drop` là đáp án.

## 11. Time / Space Complexity

Gọi `n = len(arr)`.

| Cách | Time | Space |
|---|---|---|
| Brute `O(n^3)` | `O(n^3)` | `O(n)` (do cắt list) |
| 1. Brute `O(n^2)` | `O(n^2)` | `O(1)` |
| 2. Xóa từng vị trí + Kadane | `O(n^2)` | `O(n)` |
| 3. Kadane xuôi + ngược | `O(n)` | `O(n)` |
| 4. DP mảng | `O(n)` | `O(n)` |
| 5. DP hai biến | `O(n)` | `O(1)` |

Không có khác biệt average/worst: mọi cách đều duyệt cố định theo `n`.

## 12. Bộ test

In [18]:
solvers = {
    "delete_each": solve_delete_each,
    "prefix_suffix": solve_prefix_suffix,
    "dp_arrays": solve_dp_arrays,
    "dp": solve_dp,
    "submit": Solution().maximumSum,
}
edge = [
    ([5], 5),
    ([-5], -5),
    ([-5, 3], 3),
    ([3, -5], 3),
    ([2, -100, 2], 4),
    ([-100, -1], -1),
    ([8, -1, 6, -7, -4, 5, -4, 7, -6], 17),
    ([0, 0, 0], 0),
    ([1, 2, 3], 6),
]
for arr, want in examples + edge:
    assert brute_raw(arr) == want, (arr, brute_raw(arr))
    for name, f in solvers.items():
        assert f(arr) == want, (name, arr, f(arr), want)
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [19]:
random.seed(1186)
for trial in range(1500):
    n = random.randint(1, 12)
    lo = random.choice([-10, -3, 0])
    arr = [random.randint(lo, 10) for _ in range(n)]
    want = brute_raw(arr)
    assert brute(arr) == want
    for name, f in solvers.items():
        assert f(arr) == want, (name, arr)
print("1500 test ngẫu nhiên khớp brute force")

1500 test ngẫu nhiên khớp brute force


In [20]:
random.seed(5)
big = [random.randint(-10**4, 10**4) for _ in range(10**5)]
for name in ["prefix_suffix", "dp_arrays", "dp", "submit"]:
    t = time.perf_counter(); r = solvers[name](big); print(f"{name:14s} {r} {time.perf_counter() - t:.3f}s")
allneg = [-10**4] * 10**5
assert Solution().maximumSum(allneg) == -10**4
allpos = [10**4] * 10**5
assert Solution().maximumSum(allpos) == 10**9
print("test giới hạn OK")

prefix_suffix  2651126 0.032s


dp_arrays      2651126 0.020s
dp             2651126 0.011s
submit         2651126 0.010s


test giới hạn OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Khởi tạo `drop = 0` (cho phép đoạn rỗng)

Khởi tạo `drop = 0` nghĩa là "chọn `[arr[0]]` rồi xóa nó → đoạn rỗng có tổng `0`". Nếu đồng thời khởi tạo `best = max(keep, drop)` thì giá trị `0` không hợp lệ này lọt thẳng vào đáp án:

In [21]:
def wrong_init2(arr):
    keep, drop = arr[0], 0
    best = max(keep, drop)                     # SAI lộ rõ: best = 0 với arr[0] âm
    for x in arr[1:]:
        keep, drop = max(x, keep + x), max(drop + x, keep)
        best = max(best, keep, drop)
    return best


for a in [[-1, -1, -1, -1], [-3], [-2, -7]]:
    print(a, wrong_init2(a), "đúng là", brute_raw(a))

[-1, -1, -1, -1] 0 đúng là -1
[-3] 0 đúng là -3
[-2, -7] 0 đúng là -2


**Sửa:** `drop = -inf` ở vị trí đầu tiên, `best = arr[0]`.

### Lỗi 2 — Trả về `max(0, ...)` như bài "subarray có thể rỗng"

In [22]:
def wrong_max0(arr):
    return max(0, solve_dp(arr))


print(wrong_max0([-1, -1, -1, -1]), "đúng là", -1)

0 đúng là -1


**Vì sao sai:** đề yêu cầu đoạn **không rỗng** sau khi xóa. Khác với bài 1191 (cho phép rỗng). **Sửa:** không kẹp `0`.

### Lỗi 3 — Cập nhật tuần tự `keep` trước rồi `drop` dùng `keep` mới

In [23]:
def wrong_order(arr):
    keep, drop, best = arr[0], float('-inf'), arr[0]
    for x in arr[1:]:
        keep = max(x, keep + x)
        drop = max(drop + x, keep)            # SAI: keep đã là keep[i], không phải keep[i-1]
        best = max(best, keep, drop)
    return best


print(wrong_order([1, -2, 0, 3]), "đúng là", brute_raw([1, -2, 0, 3]))

3 đúng là 4


**Vì sao sai:** khi tính `drop` ta cần lựa chọn "xóa chính `x`" = `keep[i-1]` (đoạn kết thúc ngay trước `x`). Nhưng sau khi gán tuần tự, biến `keep` đã là `keep[i]` (đoạn **chứa** `x`), nên lựa chọn "xóa `x`" biến mất — `drop` chỉ còn là "đoạn chứa `x` mà không xóa gì". Với `[1, -2, 0, 3]`: ở bước `x = -2`, `drop` phải bằng `keep[0] = 1` (xóa `-2`) nhưng lại nhận `keep[1] = -1`, nên mất đáp án `4`. Tìm thêm phản ví dụ ngẫu nhiên:

In [24]:
random.seed(13)
bad = None
for _ in range(2000):
    a = [random.randint(-5, 5) for _ in range(random.randint(1, 6))]
    if wrong_order(a) != brute_raw(a):
        bad = a; break
print("phản ví dụ:", bad, wrong_order(bad) if bad else None, brute_raw(bad) if bad else None)

phản ví dụ: [5, -1, 2, 3, 4] 13 14


**Sửa:** dùng gán đồng thời `keep, drop = max(x, keep + x), max(drop + x, keep)` hoặc cập nhật `drop` trước `keep`.

### Lỗi 4 — Chỉ xóa phần tử âm nhỏ nhất của **toàn mảng** rồi chạy Kadane

In [25]:
def wrong_global_min(arr):
    if len(arr) == 1:
        return arr[0]
    i = arr.index(min(arr))
    return max(kadane(arr), kadane(arr[:i] + arr[i + 1:]))


a = [5, -1, 5, -100, 1]
print(wrong_global_min(a), "đúng là", brute_raw(a))
a = [3, -2, 3, -50, 10, -60]
print(wrong_global_min(a), "đúng là", brute_raw(a))

10 đúng là 10
10 đúng là 14


**Vì sao sai:** phần tử cần xóa phụ thuộc **đoạn** được chọn, không phải là min toàn cục. Với `[3, -2, 3, -50, 10, -60]`: xóa `-60` (min toàn cục) chẳng giúp gì, đáp án tốt nhất là xóa `-50`: `3 - 2 + 3 + 10 = 14`. **Sửa:** dùng DP hai trạng thái.

## 14. Mẹo & nhận dạng pattern

- **"Subarray tổng lớn nhất + một ngoại lệ (xóa/đổi/lật 1 phần tử)"** → Kadane + thêm trạng thái "đã dùng quyền".
- Tổng quát: **"được dùng tối đa `k` lần một thao tác"** → DP `dp[i][j]` với `j = 0..k` lần đã dùng.
- **"Bỏ một phần tử ở giữa"** → ghép `best_end[i-1] + best_start[i+1]` (prefix/suffix).
- Trạng thái chưa hợp lệ → `-inf`, đừng dùng `0`.
- Luôn hỏi: đoạn có được rỗng không? → quyết định khởi tạo và có kẹp `0` hay không.

## 15. Biến thể

1. **LeetCode 53 — Maximum Subarray:** không xóa.
2. **Xóa tối đa `k` phần tử:** `dp[j]` = tốt nhất kết thúc tại `i` đã xóa `j` phần tử; `dp[j] = max(dp[j] + x, dp_old[j-1])`.
3. **LeetCode 1746 — Maximum Subarray Sum After One Operation:** thay một phần tử bằng `x*x` — cùng khung hai trạng thái.
4. **LeetCode 918 — Maximum Sum Circular Subarray:** Kadane max + (tổng − Kadane min).
5. **Xóa một đoạn con liên tiếp (không phải một phần tử)** → prefix best + suffix best với khoảng cách tùy ý.

In [26]:
def max_sum_k_deletions(arr, k):
    NEG = float('-inf')
    dp = [NEG] * (k + 1)
    best = NEG
    for x in arr:
        new = [NEG] * (k + 1)
        new[0] = x if dp[0] == NEG else max(x, dp[0] + x)
        for j in range(1, k + 1):
            new[j] = max(dp[j] + x, dp[j - 1])
        dp = new
        best = max(best, max(dp))
    return best


print(max_sum_k_deletions([1, -2, 0, 3], 1), max_sum_k_deletions([5, -1, -1, 5], 2), max_sum_k_deletions([5, -1, -1, 5], 1))
assert all(max_sum_k_deletions(a, 1) == brute_raw(a) for a, _ in examples + edge)

4 10 9


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | Mọi đoạn, xóa min của đoạn | `O(n^2)` | `O(1)` | Oracle để test |
| 2. Xóa từng vị trí | Kadane trên mảng bỏ `i` | `O(n^2)` | `O(n)` | Hiểu đề, input nhỏ |
| 3. Xuôi + ngược | `f[i-1] + g[i+1]` | `O(n)` | `O(n)` | Dễ giải thích, mở rộng xóa đoạn |
| 4. DP mảng | `keep[i]`, `drop[i]` | `O(n)` | `O(n)` | Debug, in bảng DP |
| 5. DP hai biến | như 4, nén bộ nhớ | `O(n)` | `O(1)` | Nộp bài |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** `maximumSum([2, -1, -1, 2])` bằng bao nhiêu?

**Bài 2 (tính tay):** lập bảng `keep`, `drop` cho `arr = [-2, 4, -3, 5]`.

**Bài 3 (điền TODO):** hoàn thiện `best_start` — mảng `g[i]` là tổng lớn nhất của đoạn bắt đầu tại `i`.

**Bài 4 (tự viết lại):** viết lại lời giải nộp bài nhưng cập nhật **tuần tự** (không gán đồng thời) mà vẫn đúng.

**Bài 5 (mini challenge):** trả về luôn **đoạn `[l, r]`** và **chỉ số bị xóa** (hoặc `None`) đạt tổng lớn nhất.

In [27]:
def best_start(arr):
    # TODO: g[i] = max tổng đoạn bắt đầu tại i
    pass


def maximum_sum_sequential(arr):
    # TODO: cập nhật drop trước, keep sau
    raise NotImplementedError


def maximum_sum_with_witness(arr):
    # TODO: trả về (tổng, l, r, chỉ_số_xóa_hoặc_None)
    raise NotImplementedError

# assert best_start([1, -2, 0, 3]) == [2, 1, 3, 3]
# assert maximum_sum_sequential([1, -2, 0, 3]) == 4

## 18. Gợi ý 3 mức

- **Mức 1:** Bắt đầu từ Kadane. Thêm một "cờ" cho biết đã dùng quyền xóa chưa.
- **Mức 2:** `drop` mới có hai nguồn: xóa `x` (lấy `keep` cũ) hoặc nối `x` vào `drop` cũ.
- **Mức 3:** Bài 4: viết `drop = max(drop + x, keep)` **trước** rồi mới `keep = max(x, keep + x)`. Bài 5: lưu kèm `(tổng, điểm bắt đầu, chỉ số xóa)` trong mỗi trạng thái, so sánh theo tổng.

## 19. Đáp án tham khảo

**Bài 1:** `3` (đoạn `[2,-1,-1,2]` xóa một `-1`: `2 - 1 + 2 = 3`).

**Bài 2:**

| i | x | keep | drop |
|---|---|---|---|
| 0 | -2 | -2 | -inf |
| 1 | 4 | 4 | -2 |
| 2 | -3 | 1 | 4 |
| 3 | 5 | 6 | 9 |

Đáp án `9` (đoạn `[4,-3,5]` xóa `-3`).

In [28]:
assert Solution().maximumSum([2, -1, -1, 2]) == 3
assert Solution().maximumSum([-2, 4, -3, 5]) == 9


def best_start(arr):
    g = arr[:]
    for i in range(len(arr) - 2, -1, -1):
        g[i] = max(arr[i], g[i + 1] + arr[i])
    return g


def maximum_sum_sequential(arr):
    keep, drop, best = arr[0], float('-inf'), arr[0]
    for x in arr[1:]:
        drop = max(drop + x, keep)   # dùng keep cũ
        keep = max(x, keep + x)
        best = max(best, keep, drop)
    return best


def maximum_sum_with_witness(arr):
    NEG = float('-inf')
    keep = (arr[0], 0, None)         # (tổng, l, chỉ_số_xóa)
    drop = (NEG, 0, None)
    best = (arr[0], 0, 0, None)
    for i in range(1, len(arr)):
        x = arr[i]
        k1 = (x, i, None)
        k2 = (keep[0] + x, keep[1], None)
        d1 = (drop[0] + x, drop[1], drop[2])
        d2 = (keep[0], keep[1], i)
        keep, drop = max(k1, k2, key=lambda t: t[0]), max(d1, d2, key=lambda t: t[0])
        for s, l, dl in (keep, drop):
            if s > best[0]:
                best = (s, l, i, dl)
    return best


def check_witness(arr, w):
    s, l, r, dl = w
    seg = [arr[j] for j in range(l, r + 1) if j != dl]
    return len(seg) >= 1 and sum(seg) == s


assert best_start([1, -2, 0, 3]) == [2, 1, 3, 3]
random.seed(19)
for _ in range(1000):
    a = [random.randint(-8, 8) for _ in range(random.randint(1, 9))]
    assert maximum_sum_sequential(a) == brute_raw(a)
    w = maximum_sum_with_witness(a)
    assert w[0] == brute_raw(a) and check_witness(a, w), (a, w)
print(maximum_sum_with_witness([1, -2, 0, 3]), "Đáp án bài tập: OK")

(4, 0, 3, 1) Đáp án bài tập: OK


## 20. Template nhớ nhanh + Checklist trước khi submit

```python
# Kadane + "được dùng 1 lần quyền X"
keep = best = a[0]
used = float('-inf')                  # trạng thái chưa hợp lệ ở phần tử đầu
for x in a[1:]:
    keep, used = max(x, keep + x), max(used + x, keep)   # keep cũ!
    best = max(best, keep, used)
return best
```

**Checklist:**
- [ ] Đoạn có được rỗng không? (bài này: **không** → không kẹp `0`).
- [ ] `drop` khởi tạo `-inf`, không phải `0`.
- [ ] `drop` mới dùng `keep` **cũ** (gán đồng thời hoặc cập nhật `drop` trước).
- [ ] Mảng 1 phần tử trả về chính nó.
- [ ] Mảng toàn số âm trả về phần tử lớn nhất.
- [ ] `O(n)` thời gian, `O(1)` bộ nhớ.